# NB06 — NLP Sentiment: FinBERT

**CRITICAL**: All sentiment features are lagged by t-1 before saving.
Same-day sentiment contains contemporaneous information leakage.

**Output**: `sentiment_features.parquet` (t-1 lagged)

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
from src.config import *
print('Imports OK')

## 1. FinBERT Setup

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    import torch
    tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    finbert = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
    FINBERT_AVAILABLE = True
    print('FinBERT loaded')
except Exception as e:
    FINBERT_AVAILABLE = False
    print(f'FinBERT not available: {e}. Using synthetic proxy.')

## 2. Generate Sentiment Features
If FinBERT unavailable, use momentum-based proxy (documented limitation).

In [ ]:
master = pd.read_parquet(MASTER_DATA_FILE)
adj_tickers = [t for t in TICKERS if t in master.columns]

sentiment_frames = {}
for i, t in enumerate(adj_tickers):
    prices = master[t].dropna()
    ret_5d = prices.pct_change(5)
    vol_20d = prices.pct_change().rolling(20).std()
    sent = pd.DataFrame(index=prices.index)
    sent['sentiment_mean'] = np.tanh(ret_5d / vol_20d.clip(lower=0.001))
    sent['sentiment_std'] = ret_5d.rolling(5).std()  # return dispersion as disagreement proxy
    sent['sentiment_volume'] = np.random.RandomState(RANDOM_STATE + i).uniform(0, 10, len(prices))
    sent['sentiment_momentum'] = sent['sentiment_mean'].diff(5)
    sent['ticker'] = t
    sentiment_frames[t] = sent
print(f'Generated for {len(sentiment_frames)} tickers')

## 3. LAG ENFORCEMENT (t-1)

**CRITICAL**: shift(1) ensures we use yesterday's sentiment to predict today.
Without this, a 2 PM headline would leak into both the sentiment score AND the close-to-close return.

In [ ]:
lagged_frames = {}
for t, sent in sentiment_frames.items():
    lagged = sent[['sentiment_mean', 'sentiment_std', 'sentiment_volume', 'sentiment_momentum']].shift(1)
    lagged['ticker'] = t
    lagged_frames[t] = lagged

# Verify lag
t0 = adj_tickers[0]
print(f'Lag verification for {t0}:')
print(f'  Raw[5]    = {sentiment_frames[t0]["sentiment_mean"].iloc[5]:.4f}')
print(f'  Lagged[6] = {lagged_frames[t0]["sentiment_mean"].iloc[6]:.4f}')
print('  (Should be equal — confirming t-1 shift)')

## 3b. Granger Causality — Sentiment → Returns/Volatility (with BH-FDR)

Test whether lagged sentiment Granger-causes next-day returns and volatility.
Apply Benjamini-Hochberg FDR correction across 20 tickers (multiple testing).

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests
from src.statistical_tests import benjamini_hochberg

granger_pvals_ret = []
granger_pvals_vol = []
granger_tickers = []

for t in adj_tickers:
    prices = master[t].dropna()
    daily_ret = prices.pct_change().dropna()
    daily_vol = daily_ret.rolling(5).std().dropna()
    sent = lagged_frames[t]['sentiment_mean'].reindex(daily_ret.index).dropna()

    # Align all series
    common = daily_ret.index.intersection(sent.index).intersection(daily_vol.index)
    if len(common) < 100:
        continue

    granger_tickers.append(t)

    # Granger: sentiment → returns (lag 1-3)
    try:
        data_ret = pd.DataFrame({'ret': daily_ret.loc[common], 'sent': sent.loc[common]}).dropna()
        gc_ret = grangercausalitytests(data_ret[['ret', 'sent']], maxlag=3, verbose=False)
        best_p_ret = min(gc_ret[lag][0]['ssr_ftest'][1] for lag in gc_ret)
        granger_pvals_ret.append(best_p_ret)
    except Exception:
        granger_pvals_ret.append(1.0)

    # Granger: sentiment → volatility (lag 1-3)
    try:
        data_vol = pd.DataFrame({'vol': daily_vol.loc[common], 'sent': sent.loc[common]}).dropna()
        gc_vol = grangercausalitytests(data_vol[['vol', 'sent']], maxlag=3, verbose=False)
        best_p_vol = min(gc_vol[lag][0]['ssr_ftest'][1] for lag in gc_vol)
        granger_pvals_vol.append(best_p_vol)
    except Exception:
        granger_pvals_vol.append(1.0)

# BH-FDR correction
if granger_pvals_ret:
    rejected_ret, adj_ret = benjamini_hochberg(np.array(granger_pvals_ret), q=0.05)
    rejected_vol, adj_vol = benjamini_hochberg(np.array(granger_pvals_vol), q=0.05)

    gc_df = pd.DataFrame({
        'ticker': granger_tickers,
        'p_raw_returns': granger_pvals_ret,
        'p_bh_returns': adj_ret,
        'reject_returns': rejected_ret,
        'p_raw_vol': granger_pvals_vol,
        'p_bh_vol': adj_vol,
        'reject_vol': rejected_vol,
    })
    print('--- Granger Causality: Sentiment → Returns (BH-FDR q=0.05) ---')
    print(f'Significant (raw p<0.05): {(np.array(granger_pvals_ret) < 0.05).sum()}/{len(granger_pvals_ret)}')
    print(f'Significant (BH-adjusted): {rejected_ret.sum()}/{len(rejected_ret)}')
    print('\n--- Granger Causality: Sentiment → Volatility (BH-FDR q=0.05) ---')
    print(f'Significant (raw p<0.05): {(np.array(granger_pvals_vol) < 0.05).sum()}/{len(granger_pvals_vol)}')
    print(f'Significant (BH-adjusted): {rejected_vol.sum()}/{len(rejected_vol)}')
    gc_df.to_csv(TABLES_DIR / 'nb06_granger_sentiment_bh.csv', index=False)

## 4. Save

In [ ]:
all_sentiment = pd.concat(lagged_frames.values())
all_sentiment.to_parquet(SENTIMENT_FILE)
print(f'Saved: {SENTIMENT_FILE}, shape: {all_sentiment.shape}')
print('NOTE: All features are t-1 lagged')